In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# 🚀 Product Launch Crew

## What We're Building

A 5-agent product launch preparation team:

| Agent | Role |
|-------|------|
| **PM Agent** | Define launch plan, positioning, and success metrics |
| **Marketing Agent** | Create go-to-market messaging and campaigns |
| **Analytics Agent** | Design tracking plan and KPI dashboard |
| **QA Agent** | Create pre-launch checklist and risk assessment |
| **Executive Summarizer** | Compile everything into an exec-ready brief |

## Key CrewAI Concept: `Process.sequential` with 5 Agents

This is the largest crew so far. Each agent's output feeds into the next,
building a complete launch plan layer by layer.

## Stack
- **CrewAI** — multi-agent pipeline
- **Ollama** — local LLM

In [ ]:
from crewai import Agent, Task, Crew, Process, LLM

llm = LLM(model="ollama/qwen3.5:9b", base_url="http://localhost:11434")
print(f"LLM ready: {llm.model}")

## Step 2 — Product Details

In [ ]:
product = """
Product: AskDocs AI
Category: Developer tools — AI-powered documentation search
What it does: Lets developers ask natural language questions about their
codebase documentation and get instant, accurate answers with source links.
Integrates with GitHub, Confluence, Notion, and ReadTheDocs.

Target users: Engineering teams at mid-size companies (50-500 devs)
Pricing: $5/dev/month (Team), $12/dev/month (Enterprise)
Launch date: 3 weeks from now
Current status: Beta with 50 users, 4.2/5 satisfaction score
Key differentiator: Works with private repos (no data leaves your infra)
Competition: Mintlify, GitBook AI, ReadMe
"""

print("Product loaded: AskDocs AI")

## Step 3 — Create 5 Agents

In [ ]:
pm_agent = Agent(
    role="Product Manager",
    goal="Define the launch strategy, positioning, and success metrics",
    backstory=(
        "You are a PM who has launched 10+ developer tools. You think in terms of "
        "user personas, value propositions, and launch milestones. You know that "
        "a great launch requires clear positioning, not just feature lists."
    ),
    llm=llm,
    verbose=True,
)

marketing_agent = Agent(
    role="Growth Marketing Lead",
    goal="Create compelling go-to-market messaging and campaign plan",
    backstory=(
        "You are a developer marketing expert. You know that devs hate "
        "traditional marketing — they want technical depth, honest comparisons, "
        "and real code examples. Your campaigns are authentic, not salesy."
    ),
    llm=llm,
    verbose=True,
)

analytics_agent = Agent(
    role="Analytics & Data Lead",
    goal="Design the tracking plan and KPI dashboard for launch measurement",
    backstory=(
        "You are a product analytics expert who has set up measurement for "
        "multiple product launches. You define clear KPIs, design event tracking, "
        "and build dashboards that show what matters — not vanity metrics."
    ),
    llm=llm,
    verbose=True,
)

qa_agent = Agent(
    role="Quality Assurance & Launch Readiness Lead",
    goal="Ensure the product is ready for launch with no show-stoppers",
    backstory=(
        "You are a QA lead who has seen launches go wrong. You create comprehensive "
        "pre-launch checklists covering technical readiness, support preparedness, "
        "documentation, and rollback plans. You're the last line of defense."
    ),
    llm=llm,
    verbose=True,
)

exec_summarizer = Agent(
    role="Executive Communications Lead",
    goal="Compile all inputs into a concise executive launch brief",
    backstory=(
        "You write executive briefs for C-level leadership. You distill complex "
        "plans into 1-page summaries with clear sections: Overview, Strategy, "
        "Timeline, Risks, and Success Criteria. No fluff, just signal."
    ),
    llm=llm,
    verbose=True,
)

print("5 agents created")

## Step 4 — Create Tasks

In [ ]:
pm_task = Task(
    description=f"""Create the product launch plan for:

{product}

Define:
1. Product positioning statement (for [target], who [need], [product] is a [category] that [benefit])
2. 3 key value propositions
3. Launch tiers: soft launch → public launch → post-launch
4. Success metrics for each tier (Week 1, Month 1, Month 3)
5. Competitive differentiators vs. Mintlify, GitBook AI, ReadMe""",
    expected_output="Product launch plan with positioning and milestones.",
    agent=pm_agent,
)

marketing_task = Task(
    description="""Based on the product positioning, create the go-to-market plan:

1. Launch announcement blog post outline
2. 5 social media posts (Twitter/X, LinkedIn, HackerNews title)
3. Product Hunt launch strategy (tagline, description, first comment)
4. Email sequence for beta users (2 emails: pre-launch teaser + launch day)
5. Developer community outreach plan (Reddit, Discord, Stack Overflow)""",
    expected_output="Complete go-to-market campaign plan.",
    agent=marketing_agent,
)

analytics_task = Task(
    description="""Design the analytics and measurement plan for launch:

1. North Star Metric: What's the ONE metric that matters most? Why?
2. KPI Framework:
   - Acquisition: signups, traffic sources, conversion rate
   - Activation: first query, time-to-value, aha moment
   - Retention: D1/D7/D30 retention, weekly active rate
   - Revenue: MRR, trial-to-paid conversion
3. Event tracking plan (10-15 key events to instrument)
4. Dashboard layout with key charts
5. Alerting rules (what triggers a red flag?)""",
    expected_output="Analytics plan with KPIs, events, and dashboard spec.",
    agent=analytics_agent,
)

qa_task = Task(
    description="""Create the pre-launch readiness checklist:

1. TECHNICAL: Load testing, error monitoring, logging, scalability
2. PRODUCT: Onboarding flow, empty states, error messages, docs
3. SUPPORT: FAQ, support channels, escalation process, SLA
4. LEGAL: Terms of service, privacy policy, data handling docs
5. ROLLBACK PLAN: What if things go wrong? Step-by-step rollback
6. RISK MATRIX: Likelihood × Impact for top 5 launch risks""",
    expected_output="Pre-launch checklist with risk matrix.",
    agent=qa_agent,
)

exec_task = Task(
    description="""Compile all the above into an executive launch brief:

Format as a 1-page executive summary:
1. OVERVIEW: What we're launching, when, for whom (3 sentences)
2. STRATEGY: Positioning + GTM approach (3 sentences)
3. KEY METRICS: North Star + 3 supporting KPIs
4. TIMELINE: Key milestones (table format)
5. TOP RISKS: Top 3 risks with mitigation
6. ASK: What the exec team needs to approve/unblock

Keep it under 500 words. Executives skim — every word must earn its place.""",
    expected_output="Concise executive launch brief under 500 words.",
    agent=exec_summarizer,
    markdown=True,
)

print("5 tasks created")

## Step 5 — Run the Crew

In [ ]:
launch_crew = Crew(
    agents=[pm_agent, marketing_agent, analytics_agent, qa_agent, exec_summarizer],
    tasks=[pm_task, marketing_task, analytics_task, qa_task, exec_task],
    process=Process.sequential,
    verbose=True,
)

print("Product launch crew assembled (5 agents)!")
result = launch_crew.kickoff()
print("\n✅ Launch brief complete!")

In [ ]:
print("📋 EXECUTIVE LAUNCH BRIEF:")
print("=" * 60)
print(result.raw)

## 🧠 Key Concepts Recap

| Concept | Implementation |
|---------|---------------|
| **5-agent crew** | PM → Marketing → Analytics → QA → Exec Summary |
| **Layer building** | Each agent adds a layer to the complete launch plan |
| **Executive distillation** | Final agent condenses everything into 500 words |
| **markdown=True** | Formatted output for the exec brief |

## 🔧 Extensions

- **Async tasks**: Run marketing and analytics in parallel
- **Planning mode**: Enable `planning=True` on the crew for auto-planning
- **File outputs**: Save each task's output to separate files
- **Guardrails**: Add word count guardrail on exec summary